# Coleta dos dados da API

In [1]:
import requests
import json

import pandas as pd
import math

## Funções para Query

Aqui, definimos as funções para, efetivamente, fazer as requisições para a API da AniList.

Começamos definindo uma constante e uma função para facilitar a escrita das requisições.

In [2]:
URL = 'https://graphql.anilist.co'

In [3]:
def embedPageQuery(query:str):
    """Coloca uma query dentro da estrutura de requisição por páginas da AniList.
        Isso evita a reescrita do mesmo código"""
    
    embededQuery =  '''
                    query ($page: Int, $perPage: Int) {
                        page: Page (page: $page, perPage: $perPage){
                    '''
    embededQuery += query
    embededQuery += "}}"
    
    return embededQuery

Agora, vamos definiar as funções para solicitar os dados à API.

`queryPages` é a função onde efetivamente é realizada a requisição. A AniList estrurua a requisição de vários dados através de "páginas" (_pages_), as quais tem no máximo 50 entradas. Portanto, se quisermos mais, vamos ter que fazer várias requisições. Portanto, esse código faz várias requisições e concatena as respostas.

O `getPages`, por outro lado, considerada que os dados da AniList vão mudar muito pouco com o tempo, então para não precisar fazer a requisição toda vez que executar o código, os dados recebidos por `queryPages` são salvos no arquivo .json, caso ele ainda não exista. No caso do arquivo já existir, o código pega os dados diretos desse arquivos, sem nem fazer a requisação à API.

In [4]:
def queryPages(url:str, query:str, total:int=50):
    """Coleta <total> entradas da query <query> em <url>"""
    
    lasting = total
    
    pages = []
    while lasting > 0:

        variables = {
            'page': math.floor((total-lasting)/50)+1,
            'perPage': lasting
        }

        print("Requisitando dados...")
        response = requests.post(url, json={'query': embedPageQuery(query), 'variables': variables})
        pages.append(json.loads(response.content)["data"]['page'])

        lasting -= 50

    jointPages = []
    key = list(pages[0].keys())[0]
    for page in pages:
        jointPages += page[key]

    return jointPages

In [5]:
def getPages(url:str, query:str, name:str, total:int=50):
    """Busca pela query nos arquivos de cache. Se não encontrar, faz a requisição."""
    
    try:
        if name == '':
            raise FileNotFoundError

        with open(f'queryCache/{name}.json', 'r') as file:
            data = json.load(file)
            
        if data['query'] != query or data['entryAmount'] != total:
            raise FileNotFoundError # Não é exatamente '_file_ not found', mas é perto o suficiente 
        
        pages = data['pages']
        print("Dados em cache!")

    except FileNotFoundError:
        pages = queryPages(url, query, total)

        data = {
            'pages': pages,
            'query': query,
            'entryAmount': total
        }

        with open(f'queryCache/{name}.json', 'w') as file:
            file.write(json.dumps(data, indent=4))

    return pages

## Requisições de Identidades

Definimos as queries a ser feitas

In [6]:
mainQuery = '''media (sort: SCORE_DESC, isAdult: false) {
                    id
                    title {
                        romaji
                        english
                        native
                    }
                    coverImage {
                        large
                    }
                    type
                    format
                    genres
                    status
                    description
                    startDate {
                        year
                        month
                        day
                    }
                    endDate {
                        year
                        month
                        day
                    }

                    episodes
                    chapters
                    volumes

                    characters (sort: FAVOURITES_DESC, page: 1, perPage: 25) {
                        edges {
                            voiceActors {
                                id
                            }
                            node{
                                id
                                name {
                                    full
                                }
                                image {
                                    large
                                }
                                description
                                gender
                                age
                            }
                        }
                    }

                    studios (sort: FAVOURITES_DESC) {
                        edges {
                          isMain
                          node{
                            id
                            name
                            }
                        }
                    }

                    staff (sort: FAVOURITES_DESC, page: 1, perPage: 25) {
                      edges{
                        role
                        node{
                            id
                            image {
                                large
                            }
                            name {
                                full
                            }
                            description
                            gender
                            age
                            primaryOccupations
                            languageV2
                        }
                      }
                    }
                }
            '''

In [7]:
# escolhi um usuário qualquer que assistiu Frieren para ser o pivot dos usuários da nossa base
userQuery = '''followers (sort: WATCHED_TIME, userId: 449691){
                id
                name
                avatar {
                    large
                }
                createdAt
                }
            '''

Organizamos as queries a serem feitas junto ao nome do seu arquivo e a quantidade de elementos para aquela query

In [8]:
queries = {
    'main':    (mainQuery,    500),
    'user':     (userQuery,    50)
}

Fazermos a requisição

In [9]:
dfs = {}
for name, (query, total) in queries.items():
    print(f"\n-- {name.capitalize()} --")
    pages = getPages(URL, query, name, total)
    dfs[name] = pd.DataFrame(pages)


-- Main --
Dados em cache!

-- User --
Dados em cache!


## Retorno das queries

In [10]:
dfs['main']

,id,title,coverImage,type,format,genres,status,description,startDate,endDate,episodes,chapters,volumes,characters,studios,staff
0,30002,"{'romaji': 'Berserk', 'english': 'Berserk', 'n...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Drama, Fantasy, Horror, Ps...",RELEASING,"His name is Guts, the Black Swordsman, a feare...","{'year': 1989, 'month': 8, 'day': 25}","{'year': None, 'month': None, 'day': None}",NaN,NaN,NaN,"{'edges': [{'voiceActors': [], 'node': {'id': ...",{'edges': []},"{'edges': [{'role': 'Story & Art (vols 1-41)',..."
1,31706,{'romaji': 'JoJo no Kimyou na Bouken: Steel Ba...,{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Drama, Mystery, Supernatural]",FINISHED,Riders from around the world gather in the Wil...,"{'year': 2004, 'month': 1, 'day': 19}","{'year': 2011, 'month': 4, 'day': 19}",NaN,95.0,24.0,"{'edges': [{'voiceActors': [], 'node': {'id': ...",{'edges': []},"{'edges': [{'role': 'Story & Art', 'node': {'i..."
2,30656,"{'romaji': 'Vagabond', 'english': 'Vagabond', ...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Drama]",RELEASING,"At seventeen years of age, Miyamoto Musashi--s...","{'year': 1998, 'month': 9, 'day': 3}","{'year': None, 'month': None, 'day': None}",NaN,NaN,NaN,"{'edges': [{'voiceActors': [], 'node': {'id': ...",{'edges': []},"{'edges': [{'role': 'Story & Art', 'node': {'i..."
3,30013,"{'romaji': 'ONE PIECE', 'english': 'One Piece'...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,MANGA,"[Action, Adventure, Comedy, Fantasy]",RELEASING,"As a child, Monkey D. Luffy was inspired to be...","{'year': 1997, 'month': 7, 'day': 22}","{'year': None, 'month': None, 'day': None}",NaN,NaN,NaN,"{'edges': [{'voiceActors': [], 'node': {'id': ...",{'edges': []},"{'edges': [{'role': 'Story & Art', 'node': {'i..."
4,154587,"{'romaji': 'Sousou no Frieren', 'english': 'Fr...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Adventure, Drama, Fantasy]",FINISHED,The adventure is over but life goes on for an ...,"{'year': 2023, 'month': 9, 'day': 29}","{'year': 2024, 'month': 3, 'day': 22}",28.0,NaN,NaN,"{'edges': [{'voiceActors': [{'id': 112215}, {'...","{'edges': [{'isMain': True, 'node': {'id': 11,...",{'edges': [{'role': 'Theme Song Performance (O...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,100675,{'romaji': 'Saenai Heroine no Sodatekata Fine'...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Comedy, Romance, Slice of Life]",FINISHED,"After Eriri and Utaha leave Blessing Software,...","{'year': 2019, 'month': 10, 'day': 26}","{'year': 2019, 'month': 10, 'day': 26}",1.0,NaN,NaN,"{'edges': [{'voiceActors': [{'id': 107711}, {'...","{'edges': [{'isMain': True, 'node': {'id': 622...","{'edges': [{'role': 'Original Story', 'node': ..."
496,108428,{'romaji': 'Kage no Jitsuryokusha ni Naritakut...,{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,NOVEL,"[Action, Comedy, Fantasy]",RELEASING,One big fat lie and a few twisted truths.\n<br...,"{'year': 2018, 'month': 11, 'day': 5}","{'year': None, 'month': None, 'day': None}",NaN,NaN,NaN,"{'edges': [{'voiceActors': [], 'node': {'id': ...",{'edges': []},"{'edges': [{'role': 'Story', 'node': {'id': 13..."
497,108430,"{'romaji': 'Given', 'english': 'given', 'nativ...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Drama, Music, Romance, Slice of Life]",FINISHED,"Tightly clutching his Gibson guitar, Mafuyu Sa...","{'year': 2019, 'month': 7, 'day': 12}","{'year': 2019, 'month': 9, 'day': 20}",11.0,NaN,NaN,"{'edges': [{'voiceActors': [{'id': 123609}, {'...","{'edges': [{'isMain': True, 'node': {'id': 456...","{'edges': [{'role': 'Original Creator', 'node'..."
498,113891,"{'romaji': 'Souyaku Toaru Majutsu no Index', '...",{'large': 'https://s4.anilist.co/file/anilistc...,MANGA,NOVEL,"[Action, Comedy, Fantasy, Sci-Fi]",RELEASING,Science and magic exist side by side in this w...,

In [11]:
dfs['main']['characters']

0      {'edges': [{'voiceActors': [], 'node': {'id': ...
1      {'edges': [{'voiceActors': [], 'node': {'id': ...
2      {'edges': [{'voiceActors': [], 'node': {'id': ...
3      {'edges': [{'voiceActors': [], 'node': {'id': ...
4      {'edges': [{'voiceActors': [{'id': 112215}, {'...
                             ...                        
495    {'edges': [{'voiceActors': [{'id': 107711}, {'...
496    {'edges': [{'voiceActors': [], 'node': {'id': ...
497    {'edges': [{'voiceActors': [{'id': 123609}, {'...
498    {'edges': [{'voiceActors': [], 'node': {'id': ...
499    {'edges': [{'voiceActors': [{'id': 95002}], 'n...
Name: characters, Length: 500, dtype: object

In [12]:
keys = [('characters', 'character'), ('studios', 'studio'), ('staff', 'staff')]

for (oldKey, newKey) in keys:
    temp = dfs['main'][oldKey].apply(lambda x: pd.DataFrame(x['edges']))
    temp = pd.concat(temp.tolist()).reset_index()
    rows = temp['node'].apply(lambda x: x.values())
    temp = pd.DataFrame(data=list(rows), columns=temp['node'].loc[0].keys())
    dfs[newKey] = temp

In [13]:
dfs['staff']

,id,image,name,description,gender,age,primaryOccupations,languageV2
0,96868,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kentarou Miura'},[Wiki (jp)](https://ja.wikipedia.org/wiki/%E4%...,Male,54.0,[Mangaka],Japanese
1,101748,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kouji Mori'},[Twitter](https://twitter.com/morichankenchan)...,Male,59.0,[Mangaka],Japanese
2,220675,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Drik Sada'},[Twitter](https://twitter.com/driksada),NaN,NaN,[Translator],Portuguese
3,143994,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Studio Gaga'},Studio Gaga is a Japanese manga production stu...,NaN,NaN,[],Japanese
4,219478,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Paweł Dybała'},[Twitter](https://twitter.com/pawel_dybala) <b...,Male,45.0,[Translator],Polish
...,...,...,...,...,...,...,...,...
7722,148930,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Yuuji Higa'},[Twitter](https://x.com/yoraku_enishi)\n\nYuuj...,NaN,NaN,[Producer],Japanese
7723,100389,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kazuma Kamachi'},[Official website](http://dengekibunko.jp/auth...,Male,NaN,[Writer],Japanese
7724,101795,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kiyotaka Haimura'},[Website](http://r-s.sakura.ne.jp/) | [Twitter...,NaN,53.0,[Illustrator],Japanese
7725,96904,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Hideaki Sorachi'},Hideaki Sorachi is a Japanese manga artist bes...,Male,47.0,[Mangaka],Japanese


In [14]:
# bad and stinky but it works so let's roll with it
for (oldKey, newKey) in keys:
    relation = f'{newKey}_midia'
    dfs[relation] = pd.DataFrame(columns=['id_media', f'id_{newKey}'])

    for (i, user) in dfs['main'].iterrows():
        edges = pd.DataFrame(user[oldKey]["edges"])
        if edges.empty:
            continue
        nodes = list(edges['node'])
        for node in nodes:
            dfs[relation].loc[len(dfs[relation])] = [user['id'], node['id']]

In [15]:
dfs['character_midia']

,id_media,id_character
0,30002,422
1,30002,424
2,30002,423
3,30002,5060
4,30002,7857
...,...,...
9878,126819,174748
9879,126819,123206
9880,126819,7928
9881,126819,18016


In [16]:
dfs['studio_midia']

,id_media,id_studio
0,154587,11
1,154587,17
2,154587,245
3,154587,53
4,154587,62
...,...,...
1142,20811,858
1143,20811,10
1144,100675,6222
1145,108430,456


In [17]:
dfs['staff_midia']

,id_media,id_staff
0,30002,96868
1,30002,101748
2,30002,220675
3,30002,143994
4,30002,219478
...,...,...
7722,108430,148930
7723,113891,100389
7724,113891,101795
7725,126819,96904


In [18]:
dfs['character_va'] = pd.DataFrame(columns=['character_id', 'va_id'])
for (i, user) in dfs['main'].iterrows():
    edges = pd.DataFrame(user['characters']["edges"])
    if edges.empty:
        continue
    nodes = edges['node']
    vaList   = edges['voiceActors']
    for vas in vaList:
        for va in vas:
            dfs['character_va'].loc[len(dfs['character_va'])] = [node['id'], va['id']]

In [19]:
dfs['character_va']

,character_id,va_id
0,182969,112215
1,182969,119697
2,182969,116893
3,182969,185758
4,182969,182231
...,...,...
20999,182969,95094
21000,182969,95103
21001,182969,95084
21002,182969,95137


In [20]:
dfs['studio_midia']['isMain'] = False
i=0
for (i, user) in dfs['main'].iterrows():
        subDfs = pd.DataFrame(user['studios']['edges'])
        for (j, subdf) in subDfs.iterrows():
            if subdf.empty:
                  continue
            dfs['studio_midia'].loc[i, 'isMain'] = subdf['isMain']
            i+=1

In [21]:
dfs['studio_midia']

,id_media,id_studio,isMain
0,154587,11,False
1,154587,17,False
2,154587,245,False
3,154587,53,False
4,154587,62,True
...,...,...,...
1142,20811,858,False
1143,20811,10,False
1144,100675,6222,False
1145,108430,456,False


In [22]:
dfs['staff_midia']['role'] = None
i=0
for (i, user) in dfs['main'].iterrows():
        edges = pd.DataFrame(user['staff']['edges'])
        for (j, edge) in edges.iterrows():
            if edge.empty:
                  continue
            dfs['staff_midia'].loc[i, 'role'] = edge['role']
            i+=1

In [23]:
dfs['main']['staff']

0      {'edges': [{'role': 'Story & Art (vols 1-41)',...
1      {'edges': [{'role': 'Story & Art', 'node': {'i...
2      {'edges': [{'role': 'Story & Art', 'node': {'i...
3      {'edges': [{'role': 'Story & Art', 'node': {'i...
4      {'edges': [{'role': 'Theme Song Performance (O...
                             ...                        
495    {'edges': [{'role': 'Original Story', 'node': ...
496    {'edges': [{'role': 'Story', 'node': {'id': 13...
497    {'edges': [{'role': 'Original Creator', 'node'...
498    {'edges': [{'role': 'Story', 'node': {'id': 10...
499    {'edges': [{'role': 'Original Creator', 'node'...
Name: staff, Length: 500, dtype: object

In [24]:
dfs['character_va']['isMain'] = False
i=0
for (i, user) in dfs['main'].iterrows():
        subDfs = pd.DataFrame(user['studios']['edges'])
        for (j, subdf) in subDfs.iterrows():
            if subdf.empty:
                  continue
            dfs['studio_midia'].loc[i, 'isMain'] = subdf['isMain']
            i+=1

In [25]:
dfs['staff']

,id,image,name,description,gender,age,primaryOccupations,languageV2
0,96868,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kentarou Miura'},[Wiki (jp)](https://ja.wikipedia.org/wiki/%E4%...,Male,54.0,[Mangaka],Japanese
1,101748,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kouji Mori'},[Twitter](https://twitter.com/morichankenchan)...,Male,59.0,[Mangaka],Japanese
2,220675,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Drik Sada'},[Twitter](https://twitter.com/driksada),NaN,NaN,[Translator],Portuguese
3,143994,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Studio Gaga'},Studio Gaga is a Japanese manga production stu...,NaN,NaN,[],Japanese
4,219478,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Paweł Dybała'},[Twitter](https://twitter.com/pawel_dybala) <b...,Male,45.0,[Translator],Polish
...,...,...,...,...,...,...,...,...
7722,148930,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Yuuji Higa'},[Twitter](https://x.com/yoraku_enishi)\n\nYuuj...,NaN,NaN,[Producer],Japanese
7723,100389,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kazuma Kamachi'},[Official website](http://dengekibunko.jp/auth...,Male,NaN,[Writer],Japanese
7724,101795,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kiyotaka Haimura'},[Website](http://r-s.sakura.ne.jp/) | [Twitter...,NaN,53.0,[Illustrator],Japanese
7725,96904,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Hideaki Sorachi'},Hideaki Sorachi is a Japanese manga artist bes...,Male,47.0,[Mangaka],Japanese


In [26]:
midia = dfs['main']
del midia['characters']
del midia['studios']
del midia['staff']

dfs['anime'] = dfs['main'][dfs['main']['type'] == 'ANIME']
dfs['manga'] = dfs['main'][dfs['main']['type'] == 'MANGA']

del dfs['anime']['volumes']
del dfs['anime']['chapters']
del dfs['manga']['episodes']

dfs['anime']

,id,title,coverImage,type,format,genres,status,description,startDate,endDate,episodes
4,154587,"{'romaji': 'Sousou no Frieren', 'english': 'Fr...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Adventure, Drama, Fantasy]",FINISHED,The adventure is over but life goes on for an ...,"{'year': 2023, 'month': 9, 'day': 29}","{'year': 2024, 'month': 3, 'day': 22}",28.0
6,114129,"{'romaji': 'Gintama: THE FINAL', 'english': 'G...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Action, Comedy, Drama, Sci-Fi]",FINISHED,Two years have passed following the Tendoshuu'...,"{'year': 2021, 'month': 1, 'day': 8}","{'year': 2021, 'month': 1, 'day': 8}",1.0
7,20996,"{'romaji': 'Gintama°', 'english': 'Gintama Sea...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Comedy, Drama, Sci-Fi]",FINISHED,"Gintoki, Shinpachi, and Kagura return as the f...","{'year': 2015, 'month': 4, 'day': 8}","{'year': 2016, 'month': 3, 'day': 30}",51.0
10,189046,{'romaji': 'Re:Zero kara Hajimeru Isekai Seika...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Drama, Fantasy, Psychologi...",FINISHED,"In the deadly battle at Priestella, Subaru and...","{'year': 2026, 'month': 4, 'day': 8}","{'year': 2026, 'month': 9, 'day': 30}",19.0
12,171627,"{'romaji': 'Chainsaw Man: Reze-hen', 'english'...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Action, Drama, Horror, Romance, Supernatural]",FINISHED,"Denji became “Chainsaw Man”, a boy with a devi...","{'year': 2025, 'month': 9, 'day': 19}","{'year': 2025, 'month': 9, 'day': 19}",1.0
...,...,...,...,...,...,...,...,...,...,...,...
489,20799,{'romaji': 'JoJo no Kimyou na Bouken: Stardust...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Drama, Supernatural]",FINISHED,"After 30 days of travel across the world, plag...","{'year': 2015, 'month': 1, 'day': 11}","{'year': 2015, 'month': 6, 'day': 20}",24.0
490,20811,{'romaji': 'Shingeki no Kyojin Gaiden: Kuinaki...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,OVA,"[Action, Drama, Fantasy]",FINISHED,This prequel to megahit Attack on Titan answer...,"{'year': 2014, 'month': 12, 'day': 9}","{'year': 2015, 'month': 4, 'day': 9}",2.0
495,100675,{'romaji': 'Saenai Heroine no Sodatekata Fine'...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Comedy, Romance, Slice of Life]",FINISHED,"After Eriri and Utaha leave Blessing Software,...","{'year': 2019, 'month': 10, 'day': 26}","{'year': 2019, 'month': 10, 'day': 26}",1.0
497,108430,"{'romaji': 'Given', 'english': 'given', 'nativ...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Drama, Music, Romance, Slice of Life]",FINISHED,"Tightly clutching his Gibson guitar, Mafuyu Sa...","{'year': 2019, 'month': 7, 'day': 12}","{'year': 2019, 'month': 9, 'day': 20}",11.0


In [27]:
dfs['user']

,id,name,avatar,createdAt
0,312878,armennnn,{'large': 'https://s4.anilist.co/file/anilistc...,1564180699
1,346515,MolhoShouyou,{'large': 'https://s4.anilist.co/file/anilistc...,1569935898
2,448984,lotusmilktea,{'large': 'https://s4.anilist.co/file/anilistc...,1586111998
3,482510,39teeth,{'large': 'https://s4.anilist.co/file/anilistc...,1588421801
4,489682,SpamAcnt,{'large': 'https://s4.anilist.co/file/anilistc...,1588918728
5,503078,nanimonoda,{'large': 'https://s4.anilist.co/file/anilistc...,1589865613
6,503138,Gant,{'large': 'https://s4.anilist.co/file/anilistc...,1589869918
7,510693,jaena,{'large': 'https://s4.anilist.co/file/anilistc...,1590428432
8,543506,Nebu,{'large': 'https://s4.anilist.co/file/anilistc...,1592783048
9,551300,remsenpaii,{'large': 'https://s4.anilist.co/file/anilistc...,1593345002


In [28]:
followingQuery = '''
                    followers (sort: WATCHED_TIME, userId: %d){
                        id
                        }
                '''

In [29]:
dfs['user_user'] = pd.DataFrame(columns=['followed', 'follower'])
for (i, user) in dfs['user'].iterrows():
    query = followingQuery % user['id']
    pages = getPages(URL, query, f'followers/user{user["id"]}followers')
    df = pd.DataFrame(pages)
    followers = pd.merge(df, dfs['user'], on='id') # Seguidores presentes nos nossos dados
    for (j, follower) in followers.iterrows():
        idFollower = user['id']
        idFollowed = follower['id']
        dfs['user_user'].loc[len(dfs['user_user'])] = [idFollower, idFollowed]
dfs['user_user']

Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!
Dados em cache!


,followed,follower
0,346515,600603
1,503078,600603
2,553430,543506
3,578383,574881
4,578383,600603
5,683219,503138
6,688689,503138
7,800984,269225
8,800984,205601
9,830633,551300


## Expansão de colunas compostas

Na resposta da API, muitas colunas são compostas, sendo seus valores listas ou dicionários. Vamos expandí-las em tabelas próprias de acordo com o modelo lógico.

### Títulos

In [30]:
dfs['title'] = pd.DataFrame()

animeTitles = pd.DataFrame()
mangaTitles = pd.DataFrame()

animeTitles['id_midia'] = dfs['anime']['id']
mangaTitles['id_midia'] = dfs['anime']['id']

titleTypes = ['romaji', 'english', 'native']

for titleType in titleTypes:
    animeTitles[titleType] = dfs['anime']['title'].apply(lambda x: x[titleType])
    mangaTitles[titleType] = dfs['manga']['title'].apply(lambda x: x[titleType])


dfs['title'] = pd.concat([animeTitles, mangaTitles])

dfs['title']

,id_midia,romaji,english,native
4,154587,Sousou no Frieren,Frieren: Beyond Journey’s End,葬送のフリーレン
6,114129,Gintama: THE FINAL,Gintama: THE VERY FINAL,銀魂 THE FINAL
7,20996,Gintama°,Gintama Season 3,銀魂゜
10,189046,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,Re:ZERO -Starting Life in Another World- Season 4,Re:ゼロから始める異世界生活 4th season
12,171627,Chainsaw Man: Reze-hen,Chainsaw Man – The Movie: Reze Arc,チェンソーマン レゼ篇
...,...,...,...,...
489,20799,NaN,NaN,NaN
490,20811,NaN,NaN,NaN
495,100675,NaN,NaN,NaN
497,108430,NaN,NaN,NaN


In [31]:
del dfs['anime']['title']
del dfs['manga']['title']

### Gêneros

In [32]:
compondGenres = pd.concat([dfs['anime'][['id', 'genres']], dfs['manga'][['id', 'genres']]])

dfs['genre'] = pd.DataFrame(columns=['id_midia', 'name'])

# bad and stinky code but god has punished me for my hubris and my work is never finished 
# so I don't have the time to fix it
for id, occupations in compondGenres.values:
    for genre in occupations:
        dfs['genre'].loc[len(dfs['genre'])] = [id, genre]

dfs['genre']

,id_midia,name
0,154587,Adventure
1,154587,Drama
2,154587,Fantasy
3,114129,Action
4,114129,Comedy
...,...,...
1850,108428,Fantasy
1851,113891,Action
1852,113891,Comedy
1853,113891,Fantasy


In [33]:
del dfs['anime']['genres']
del dfs['manga']['genres']

### Ocupações

In [34]:
compondOccupations = dfs['staff'][['id', 'primaryOccupations']]

dfs['occupation'] = pd.DataFrame(columns=['id_staff', 'occupation'])

# bad and stinky code but god has punished me for my hubris and my work is never finished 
# so I don't have the time to fix it
for id, occupations in compondOccupations.values:
    for occupation in occupations:
        dfs['occupation'].loc[len(dfs['occupation'])] = [id, occupation]

dfs['occupation']

,id_staff,occupation
0,96868,Mangaka
1,101748,Mangaka
2,220675,Translator
3,219478,Translator
4,214718,Translator
...,...,...
10631,148930,Producer
10632,100389,Writer
10633,101795,Illustrator
10634,96904,Mangaka


In [35]:
partialStaff = dfs['staff']
partialStaff['id_staff'] = partialStaff['id']
mergedStaff = pd.merge(dfs['occupation'], partialStaff, on='id_staff')

voiceActors = mergedStaff[mergedStaff['occupation'] == 'Voice Actor']

dfs['va']    = voiceActors[['id_staff', 'languageV2']]
dfs['va']

,id_staff,languageV2
605,95472,English
606,95472,English
607,95472,English
608,95472,English
609,95472,English
...,...,...
101549,127741,English
101550,127741,English
101560,128533,English
101571,195092,Portuguese


In [36]:
del dfs['staff']['primaryOccupations']

### Campos atômicos dentro de listas e dicionários

#### Imagens

In [37]:
extractImage = lambda x: x['large']

dfs['anime']['coverImage']  = dfs['anime']['coverImage'].apply(extractImage)
dfs['manga']['coverImage']  = dfs['manga']['coverImage'].apply(extractImage)
dfs['staff']['image']       = dfs['staff']['image'].apply(extractImage)
dfs['user']['avatar']       = dfs['user']['avatar'].apply(extractImage)
dfs['character']['image']   = dfs['character']['image'].apply(extractImage)

In [38]:
dfs['character']['image']

0       https://s4.anilist.co/file/anilistcdn/characte...
1       https://s4.anilist.co/file/anilistcdn/characte...
2       https://s4.anilist.co/file/anilistcdn/characte...
3       https://s4.anilist.co/file/anilistcdn/characte...
4       https://s4.anilist.co/file/anilistcdn/characte...
                              ...                        
9878    https://s4.anilist.co/file/anilistcdn/characte...
9879    https://s4.anilist.co/file/anilistcdn/characte...
9880    https://s4.anilist.co/file/anilistcdn/characte...
9881    https://s4.anilist.co/file/anilistcdn/characte...
9882    https://s4.anilist.co/file/anilistcdn/characte...
Name: image, Length: 9883, dtype: str

#### Nomes

In [39]:
extractName = lambda x: x['full']

dfs['character']['name'] = dfs['character']['name'].apply(extractName)
dfs['staff']['name'] = dfs['staff']['name'].apply(extractName)

In [40]:
dfs['staff']['name']

0         Kentarou Miura
1             Kouji Mori
2              Drik Sada
3            Studio Gaga
4           Paweł Dybała
              ...       
7722          Yuuji Higa
7723      Kazuma Kamachi
7724    Kiyotaka Haimura
7725     Hideaki Sorachi
7726    Hotaru Morishige
Name: name, Length: 7727, dtype: str

## Tipagem

Alguns tipos vieram inapropriados, como float quando deveria ser inteiro ou datas quebradas de formas estranhas. Vamos ajeitas isso!

### Datas

In [41]:
def dict2timestamp(x):
    try:
        return pd.Timestamp(year=x['year'], month=x['month'], day=x['day'])
    except:
        return None

dfs['anime']['startDate'] = dfs['anime']['startDate'].apply(dict2timestamp)
dfs['anime']['endDate']   = dfs['anime']['endDate'].apply(dict2timestamp)

dfs['manga']['startDate'] = dfs['manga']['startDate'].apply(dict2timestamp)
dfs['manga']['endDate']   = dfs['manga']['endDate'].apply(dict2timestamp)

In [42]:
dfs['anime']['startDate']

4     2023-09-29
6     2021-01-08
7     2015-04-08
10    2026-04-08
12    2025-09-19
         ...    
489   2015-01-11
490   2014-12-09
495   2019-10-26
497   2019-07-12
499   2021-01-15
Name: startDate, Length: 258, dtype: datetime64[us]

In [43]:
dfs['user']['createdAt'] = pd.to_datetime(dfs['user']['createdAt'], unit='s')

In [44]:
dfs['user']['createdAt']

0    2019-07-26 22:38:19
1    2019-10-01 13:18:18
2    2020-04-05 18:39:58
3    2020-05-02 12:16:41
4    2020-05-08 06:18:48
5    2020-05-19 05:20:13
6    2020-05-19 06:31:58
7    2020-05-25 17:40:32
8    2020-06-21 23:44:08
9    2020-06-28 11:50:02
10   2020-06-29 23:03:28
11   2020-07-11 19:10:02
12   2020-07-13 12:33:49
13   2020-07-19 19:55:51
14   2020-07-26 04:00:11
15   2020-09-07 19:53:51
16   2020-09-11 10:04:09
17   2020-09-28 18:22:11
18   2020-11-12 19:30:03
19   2020-11-30 18:24:25
20   2020-12-11 18:46:24
21   2020-12-28 13:44:11
22   2021-03-15 14:11:14
23   2021-03-31 14:02:53
24   2021-04-04 10:02:44
25   2021-06-03 18:53:37
26   2021-09-03 17:04:21
27   2021-10-20 12:58:19
28   2025-10-25 05:23:44
29   2018-09-21 19:59:17
30   2019-04-21 22:56:54
31   2018-06-13 09:58:17
32   2019-09-11 10:03:48
33   2019-01-20 22:40:40
34   2018-11-09 15:51:57
35   2018-06-02 02:04:29
Name: createdAt, dtype: datetime64[s]

### Numeros

In [45]:
dfs['anime']['episodes'] = dfs['anime']['episodes'].astype('Int64')

In [46]:
dfs['manga']['chapters'] = dfs['manga']['chapters'].astype('Int64')
dfs['manga']['volumes'] = dfs['manga']['chapters'].astype('Int64')

In [47]:
dfs['staff']['age'] = dfs['staff']['age'].astype('Int64')

## Requisição de Relacionamentos

In [48]:
userIds = dfs['user']['id'].tolist()
userIds

[312878,
 346515,
 448984,
 482510,
 489682,
 503078,
 503138,
 510693,
 543506,
 551300,
 553430,
 574881,
 578383,
 589245,
 600603,
 683219,
 688689,
 718296,
 800984,
 830633,
 851265,
 880729,
 5192495,
 5243660,
 5258871,
 5390868,
 5552465,
 5616002,
 7544839,
 121034,
 269225,
 132282,
 300115,
 228269,
 205601,
 130474]

In [49]:
mediaListQuery = '''
                    mediaList(userId_in:''' + str(userIds) + '''){
                        id
                        media {
                            id
                        }
                        userId
                        status
                        score
                        progress
                        startedAt {
                            day
                            month
                            year
                        }
                    }
                '''

In [50]:
queries = {
    'mediaList':    (mediaListQuery, 1000), # Só tem 991 entradas ao todo
}

Fazermos a requisição

In [51]:
for name, (query, total) in queries.items():
    print(f"\n-- {name.capitalize()} --")
    pages = getPages(URL, query, name, total)
    dfs[name] = pd.DataFrame(pages)


-- Medialist --
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...


In [52]:
dfs['mediaList']

,id,media,userId,status,score,progress,startedAt
0,25339997,{'id': 16932},130474,COMPLETED,60,1,"{'day': 22, 'month': 12, 'year': 2019}"
1,25396709,{'id': 87306},130474,COMPLETED,70,19,"{'day': 26, 'month': 11, 'year': 2016}"
2,28232026,{'id': 101806},130474,PAUSED,0,6,"{'day': None, 'month': None, 'year': None}"
3,29658296,{'id': 99474},130474,COMPLETED,70,12,"{'day': 22, 'month': 5, 'year': 2019}"
4,30978668,{'id': 102028},130474,COMPLETED,80,8,"{'day': 19, 'month': 6, 'year': 2022}"
...,...,...,...,...,...,...,...
995,49217495,{'id': 20992},205601,COMPLETED,95,25,"{'day': 4, 'month': 10, 'year': 2015}"
996,49217506,{'id': 21698},205601,COMPLETED,90,10,"{'day': 8, 'month': 10, 'year': 2016}"
997,49217509,{'id': 15613},205601,COMPLETED,95,13,"{'day': None, 'month': None, 'year': None}"
998,49217510,{'id': 18055},205601,COMPLETED,95,13,"{'day': None, 'month': None, 'year': None}"


In [53]:
dfs['mediaList']['media_id'] = dfs['mediaList']['media'].apply(lambda x: x['id'])

In [54]:
dfs['mediaList']['startedAt'] = dfs['mediaList']['startedAt'].apply(dict2timestamp)

In [55]:
del dfs['mediaList']['media']

In [56]:
dfs['mediaList']

,id,userId,status,score,progress,startedAt,media_id
0,25339997,130474,COMPLETED,60,1,2019-12-22,16932
1,25396709,130474,COMPLETED,70,19,2016-11-26,87306
2,28232026,130474,PAUSED,0,6,NaT,101806
3,29658296,130474,COMPLETED,70,12,2019-05-22,99474
4,30978668,130474,COMPLETED,80,8,2022-06-19,102028
...,...,...,...,...,...,...,...
995,49217495,205601,COMPLETED,95,25,2015-10-04,20992
996,49217506,205601,COMPLETED,90,10,2016-10-08,21698
997,49217509,205601,COMPLETED,95,13,NaT,15613
998,49217510,205601,COMPLETED,95,13,NaT,18055


## Dados limpos

In [57]:
dfs['anime']

,id,coverImage,type,format,status,description,startDate,endDate,episodes
4,154587,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,The adventure is over but life goes on for an ...,2023-09-29,2024-03-22,28
6,114129,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,MOVIE,FINISHED,Two years have passed following the Tendoshuu'...,2021-01-08,2021-01-08,1
7,20996,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"Gintoki, Shinpachi, and Kagura return as the f...",2015-04-08,2016-03-30,51
10,189046,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"In the deadly battle at Priestella, Subaru and...",2026-04-08,2026-09-30,19
12,171627,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,MOVIE,FINISHED,"Denji became “Chainsaw Man”, a boy with a devi...",2025-09-19,2025-09-19,1
...,...,...,...,...,...,...,...,...,...
489,20799,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"After 30 days of travel across the world, plag...",2015-01-11,2015-06-20,24
490,20811,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,OVA,FINISHED,This prequel to megahit Attack on Titan answer...,2014-12-09,2015-04-09,2
495,100675,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,MOVIE,FINISHED,"After Eriri and Utaha leave Blessing Software,...",2019-10-26,2019-10-26,1
497,108430,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"Tightly clutching his Gibson guitar, Mafuyu Sa...",2019-07-12,2019-09-20,11


In [58]:
dfs['manga']

,id,coverImage,type,format,status,description,startDate,endDate,chapters,volumes
0,30002,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,RELEASING,"His name is Guts, the Black Swordsman, a feare...",1989-08-25,NaT,<NA>,<NA>
1,31706,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,Riders from around the world gather in the Wil...,2004-01-19,2011-04-19,95,95
2,30656,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,RELEASING,"At seventeen years of age, Miyamoto Musashi--s...",1998-09-03,NaT,<NA>,<NA>
3,30013,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,RELEASING,"As a child, Monkey D. Luffy was inspired to be...",1997-07-22,NaT,<NA>,<NA>
5,64053,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,It's time once again for the annual family gat...,2012-01-21,2015-06-22,42,42
...,...,...,...,...,...,...,...,...,...,...
492,36812,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,After Takashi's family moves into a new neighb...,1988-02-10,1997-11-15,366,366
493,41327,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,MANGA,FINISHED,"The story begins when Hiromichi Kojima, the st...",1998-06-18,2009-02-19,176,176
494,86406,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,NOVEL,FINISHED,I used to be a normal high school girl but in ...,2015-12-10,2022-01-08,384,384
496,108428,https://s4.anilist.co/file/anilistcdn/media/ma...,MANGA,NOVEL,RELEASING,One big fat lie and a few twisted truths.\n<br...,2018-11-05,NaT,<NA>,<NA>


In [59]:
dfs['studio']

,id,name
0,11,MADHOUSE
1,17,Aniplex
2,245,Toho
3,53,Dentsu
4,62,Shogakukan-Shueisha Productions
...,...,...
1142,858,WIT STUDIO
1143,10,Production I.G
1144,6222,CloverWorks
1145,456,Lerche


In [60]:
dfs['character']

,id,name,image,description,gender,age
0,422,Guts,https://s4.anilist.co/file/anilistcdn/characte...,Guts is the protagonist of Berserk. He is a ta...,Male,NaN
1,424,Griffith,https://s4.anilist.co/file/anilistcdn/characte...,Griffith was the founder and leader of the mer...,Male,NaN
2,423,Casca,https://s4.anilist.co/file/anilistcdn/characte...,Casca was the only female soldier in the origi...,Female,NaN
3,5060,Puck,https://s4.anilist.co/file/anilistcdn/characte...,"A wind spirit from Elfhelm, a utopia on the is...",NaN,NaN
4,7857,Farnese de Vandimion,https://s4.anilist.co/file/anilistcdn/characte...,Farnese is a noblewoman who led the Holy See's...,Female,NaN
...,...,...,...,...,...,...
9878,174748,Asaemon Ikeda,https://s4.anilist.co/file/anilistcdn/characte...,**Race:** Human\n**Affiliation:** Ikeda clan\n...,Female,NaN
9879,123206,Shimaru Saitou,https://s4.anilist.co/file/anilistcdn/characte...,The special captain of the third unit of the S...,NaN,NaN
9880,7928,Hedoro,https://s4.anilist.co/file/anilistcdn/characte...,Hedoro is a monster that befriends the Yoruzu...,NaN,NaN
9881,18016,Tetsuko Murata,https://s4.anilist.co/file/anilistcdn/characte...,__Height:__ 161 cm\n\nMurata Tetsuko is a swor...,Female,NaN


In [61]:
dfs['staff']

,id,image,name,description,gender,age,languageV2,id_staff
0,96868,https://s4.anilist.co/file/anilistcdn/staff/la...,Kentarou Miura,[Wiki (jp)](https://ja.wikipedia.org/wiki/%E4%...,Male,54,Japanese,96868
1,101748,https://s4.anilist.co/file/anilistcdn/staff/la...,Kouji Mori,[Twitter](https://twitter.com/morichankenchan)...,Male,59,Japanese,101748
2,220675,https://s4.anilist.co/file/anilistcdn/staff/la...,Drik Sada,[Twitter](https://twitter.com/driksada),NaN,<NA>,Portuguese,220675
3,143994,https://s4.anilist.co/file/anilistcdn/staff/la...,Studio Gaga,Studio Gaga is a Japanese manga production stu...,NaN,<NA>,Japanese,143994
4,219478,https://s4.anilist.co/file/anilistcdn/staff/la...,Paweł Dybała,[Twitter](https://twitter.com/pawel_dybala) <b...,Male,45,Polish,219478
...,...,...,...,...,...,...,...,...
7722,148930,https://s4.anilist.co/file/anilistcdn/staff/la...,Yuuji Higa,[Twitter](https://x.com/yoraku_enishi)\n\nYuuj...,NaN,<NA>,Japanese,148930
7723,100389,https://s4.anilist.co/file/anilistcdn/staff/la...,Kazuma Kamachi,[Official website](http://dengekibunko.jp/auth...,Male,<NA>,Japanese,100389
7724,101795,https://s4.anilist.co/file/anilistcdn/staff/la...,Kiyotaka Haimura,[Website](http://r-s.sakura.ne.jp/) | [Twitter...,NaN,53,Japanese,101795
7725,96904,https://s4.anilist.co/file/anilistcdn/staff/la...,Hideaki Sorachi,Hideaki Sorachi is a Japanese manga artist bes...,Male,47,Japanese,96904


In [62]:
dfs['title']

,id_midia,romaji,english,native
4,154587,Sousou no Frieren,Frieren: Beyond Journey’s End,葬送のフリーレン
6,114129,Gintama: THE FINAL,Gintama: THE VERY FINAL,銀魂 THE FINAL
7,20996,Gintama°,Gintama Season 3,銀魂゜
10,189046,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,Re:ZERO -Starting Life in Another World- Season 4,Re:ゼロから始める異世界生活 4th season
12,171627,Chainsaw Man: Reze-hen,Chainsaw Man – The Movie: Reze Arc,チェンソーマン レゼ篇
...,...,...,...,...
489,20799,NaN,NaN,NaN
490,20811,NaN,NaN,NaN
495,100675,NaN,NaN,NaN
497,108430,NaN,NaN,NaN


In [63]:
dfs['user']

,id,name,avatar,createdAt
0,312878,armennnn,https://s4.anilist.co/file/anilistcdn/user/ava...,2019-07-26 22:38:19
1,346515,MolhoShouyou,https://s4.anilist.co/file/anilistcdn/user/ava...,2019-10-01 13:18:18
2,448984,lotusmilktea,https://s4.anilist.co/file/anilistcdn/user/ava...,2020-04-05 18:39:58
3,482510,39teeth,https://s4.anilist.co/file/anilistcdn/user/ava...,2020-05-02 12:16:41
4,489682,SpamAcnt,https://s4.anilist.co/file/anilistcdn/user/ava...,2020-05-08 06:18:48
5,503078,nanimonoda,https://s4.anilist.co/file/anilistcdn/user/ava...,2020-05-19 05:20:13
6,503138,Gant,https://s4.anilist.co/file/anilistcdn/user/ava...,2020-05-19 06:31:58
7,510693,jaena,https://s4.anilist.co/file/anilistcdn/user/ava...,2020-05-25 17:40:32
8,543506,Nebu,https://s4.anilist.co/file/anilistcdn/user/ava...,2020-06-21 23:44:08
9,551300,remsenpaii,https://s4.anilist.co/file/anilistcdn/user/ava...,2020-06-28 11:50:02


In [64]:
dfs['genre']

,id_midia,name
0,154587,Adventure
1,154587,Drama
2,154587,Fantasy
3,114129,Action
4,114129,Comedy
...,...,...
1850,108428,Fantasy
1851,113891,Action
1852,113891,Comedy
1853,113891,Fantasy


In [65]:
dfs['occupation']

,id_staff,occupation
0,96868,Mangaka
1,101748,Mangaka
2,220675,Translator
3,219478,Translator
4,214718,Translator
...,...,...
10631,148930,Producer
10632,100389,Writer
10633,101795,Illustrator
10634,96904,Mangaka


In [66]:
dfs['va']

,id_staff,languageV2
605,95472,English
606,95472,English
607,95472,English
608,95472,English
609,95472,English
...,...,...
101549,127741,English
101550,127741,English
101560,128533,English
101571,195092,Portuguese


In [67]:
dfs['character_midia']

,id_media,id_character
0,30002,422
1,30002,424
2,30002,423
3,30002,5060
4,30002,7857
...,...,...
9878,126819,174748
9879,126819,123206
9880,126819,7928
9881,126819,18016


In [68]:
dfs['studio_midia']

,id_media,id_studio,isMain
0,154587,11,False
1,154587,17,False
2,154587,245,False
3,154587,53,False
4,154587,62,True
...,...,...,...
1142,20811,858,False
1143,20811,10,False
1144,100675,6222,False
1145,108430,456,False


In [69]:
dfs['staff_midia']

,id_media,id_staff,role
0,30002,96868,Story & Art (vols 1-41)
1,30002,101748,Story & Art
2,30002,220675,Story & Art
3,30002,143994,Story & Art
4,30002,219478,Theme Song Performance (OP1)
...,...,...,...
7722,108430,148930,None
7723,113891,100389,None
7724,113891,101795,None
7725,126819,96904,None


In [70]:
dfs['mediaList']

,id,userId,status,score,progress,startedAt,media_id
0,25339997,130474,COMPLETED,60,1,2019-12-22,16932
1,25396709,130474,COMPLETED,70,19,2016-11-26,87306
2,28232026,130474,PAUSED,0,6,NaT,101806
3,29658296,130474,COMPLETED,70,12,2019-05-22,99474
4,30978668,130474,COMPLETED,80,8,2022-06-19,102028
...,...,...,...,...,...,...,...
995,49217495,205601,COMPLETED,95,25,2015-10-04,20992
996,49217506,205601,COMPLETED,90,10,2016-10-08,21698
997,49217509,205601,COMPLETED,95,13,NaT,15613
998,49217510,205601,COMPLETED,95,13,NaT,18055


In [71]:
dfs['user_user']

,followed,follower
0,346515,600603
1,503078,600603
2,553430,543506
3,578383,574881
4,578383,600603
5,683219,503138
6,688689,503138
7,800984,269225
8,800984,205601
9,830633,551300


In [72]:
for table in dfs.keys():
    dfs[table].to_csv(f'./data/{table}.csv', index=False)

## Testes

In [73]:
anime = dfs['anime']
anime['media_id'] = anime['id']

In [74]:
pd.merge(anime, dfs['mediaList'], on='media_id')

,id_x,coverImage,type,format,status_x,description,startDate,endDate,episodes,media_id,id_y,userId,status_y,score,progress,startedAt
0,20996,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"Gintoki, Shinpachi, and Kagura return as the f...",2015-04-08,2016-03-30,51,20996,49181207,205601,COMPLETED,100,51,2015-04-08
1,5114,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"""In order for something to be obtained, someth...",2009-04-05,2010-07-04,64,5114,32264718,132282,PAUSED,0,20,2017-11-24
2,5114,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"""In order for something to be obtained, someth...",2009-04-05,2010-07-04,64,5114,49181018,205601,COMPLETED,100,64,NaT
3,9253,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,Self-proclaimed mad scientist Okabe Rintarou l...,2011-04-06,2011-09-14,24,9253,32264749,132282,PAUSED,0,21,2018-05-08
4,11061,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,A Hunter is one who travels the world doing al...,2011-10-02,2014-09-24,148,11061,43550773,132282,COMPLETED,100,148,2018-09-04
5,15417,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"While Gintoki Sakata was away, the Yorozuya fo...",2012-10-04,2013-03-28,13,15417,49181170,205601,COMPLETED,95,13,NaT
6,9969,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"After a one-year hiatus, Shinpachi Shimura ret...",2011-04-04,2012-03-26,51,9969,49181168,205601,COMPLETED,100,51,NaT
7,97889,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,After joining the resistance against the bakuf...,2017-01-09,2017-03-27,12,97889,49181178,205601,COMPLETED,100,12,2017-07-29
8,21,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,RELEASING,"Gold Roger was known as the Pirate King, the s...",1999-10-20,NaT,<NA>,21,49181432,205601,CURRENT,100,1180,2026-09-27
9,101925,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,Following the temporary retreat of the Altana ...,2018-07-09,2018-10-08,14,101925,49181197,205601,COMPLETED,100,14,2018-07-09


In [75]:
anime

,id,coverImage,type,format,status,description,startDate,endDate,episodes,media_id
4,154587,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,The adventure is over but life goes on for an ...,2023-09-29,2024-03-22,28,154587
6,114129,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,MOVIE,FINISHED,Two years have passed following the Tendoshuu'...,2021-01-08,2021-01-08,1,114129
7,20996,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"Gintoki, Shinpachi, and Kagura return as the f...",2015-04-08,2016-03-30,51,20996
10,189046,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"In the deadly battle at Priestella, Subaru and...",2026-04-08,2026-09-30,19,189046
12,171627,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,MOVIE,FINISHED,"Denji became “Chainsaw Man”, a boy with a devi...",2025-09-19,2025-09-19,1,171627
...,...,...,...,...,...,...,...,...,...,...
489,20799,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"After 30 days of travel across the world, plag...",2015-01-11,2015-06-20,24,20799
490,20811,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,OVA,FINISHED,This prequel to megahit Attack on Titan answer...,2014-12-09,2015-04-09,2,20811
495,100675,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,MOVIE,FINISHED,"After Eriri and Utaha leave Blessing Software,...",2019-10-26,2019-10-26,1,100675
497,108430,https://s4.anilist.co/file/anilistcdn/media/an...,ANIME,TV,FINISHED,"Tightly clutching his Gibson guitar, Mafuyu Sa...",2019-07-12,2019-09-20,11,108430
